# Lecture 15 examples: Wrapping up ABC: Comparisons with SMM

STAT-S681 · Simulation-Based Inference

# 1. Identifiability: insects caught in a trap

We count the insects caught each night. Let $\lambda$ be the expected number of arrivals per night and $q$ the probability of catching each arriving insect.

## The trap simulator

For each night $i$:

$$
N_i\sim\operatorname{Poisson}(\lambda),
\qquad Y_i\mid N_i\sim\operatorname{Binomial}(N_i,q).
$$

We observe the catches $Y_1,\ldots,Y_n$. We do not observe the arrivals $N_i$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(681)

def simulate_catches(lam, q, n):
    arrivals = rng.poisson(lam, n)        # N_i
    return rng.binomial(arrivals, q)      # Y_i given N_i

y_obs_300 = simulate_catches(lam=10, q=0.4, n=300)
y_obs = y_obs_300[:30]   # first 30 nights

## Fitting the trap model with ABC

Draw $(\lambda,q)$ from the prior and simulate $n$ nightly catches. Retain the parameter pair if the total simulated catch equals the observed total.

In [ ]:
n_prop = 200_000
lam = rng.uniform(0, 40, n_prop)
q = rng.uniform(0, 1, n_prop)
n = len(y_obs)
# row i holds n simulated nights at (lam[i], q[i])
arrivals = rng.poisson(lam[:, None], (n_prop, n))
catches = rng.binomial(arrivals, q[:, None])
keep = catches.sum(axis=1) == y_obs.sum()
post_trap = np.column_stack([lam[keep], q[keep]])
plt.scatter(post_trap[:, 0], post_trap[:, 1])

This is a ridge with high posterior mass.

## Step 1: Pick points along the ridge

For $\lambda$ near 5 and near 30, take the median $q$ among retained pairs with $\lambda$ within 10%.

In [ ]:
# a point on the ridge: the median q among retained pairs
# whose lambda is within 10% of lam0
def ridge_point(post, lam0):
    near = np.abs(post[:, 0] / lam0 - 1) < 0.1
    return lam0, np.median(post[near, 1])

pt_a = ridge_point(post_trap, 5)
pt_b = ridge_point(post_trap, 30)

plt.scatter(post_trap[:, 0], post_trap[:, 1], s=2, color="gray")
plt.scatter([pt_a[0], pt_b[0]], [pt_a[1], pt_b[1]], s=80)

## Step 2: Simulate from each point and compare

Simulate many nights at each point and compare the two distributions of catches directly.

In [ ]:
catch_a = simulate_catches(*pt_a, n=100_000)
catch_b = simulate_catches(*pt_b, n=100_000)

fig, ax = plt.subplots(1, 3)
ax[0].hist(catch_a, bins=np.arange(-0.5, 26))
ax[1].hist(catch_b, bins=np.arange(-0.5, 26))
# overlaid, bins of width 1
ax[2].hist(catch_a, bins=np.arange(-0.5, 26), alpha=0.5)
ax[2].hist(catch_b, bins=np.arange(-0.5, 26), alpha=0.5)

The histograms match, up to simulation noise.

# 2. Model checking: weather over several decades

We observe dry and rainy days during the same 30-day calendar period at one location, for 30 years.

## A model for the rain records

Let $X_{ij}=1$ if day $j$ of year $i$ is dry and $X_{ij}=0$ if it rains. Years are independent. Within year $i$:

$$
X_{i1}\sim\operatorname{Bernoulli}(p),
\qquad
X_{ij}\mid X_{i,j-1}\sim\operatorname{Bernoulli}\big(\rho X_{i,j-1}+(1-\rho)p\big).
$$

In [ ]:
def simulate_year(p, rho, n_days=30):
    dry = np.zeros(n_days, dtype=int)
    dry[0] = rng.random() < p
    for t in range(1, n_days):
        if rng.random() < rho:
            dry[t] = dry[t - 1]          # copy yesterday
        else:
            dry[t] = rng.random() < p    # fresh weather
    return dry

# one row per year, one column per day
def simulate_weather(p, rho, n_years=30, n_days=30):
    return np.array([simulate_year(p, rho, n_days)
                     for _ in range(n_years)])

## The observed records

The observed 30 × 30 matrix: one row per year, one column per day, 1 = dry.

In [ ]:
x_obs = np.loadtxt("weather-records.csv", delimiter=",", dtype=int)

## ABC for the weather records

Summaries: the fraction of all days that are dry, and the fraction of within-year transitions that switch weather. Retain the $(p,\rho)$ with the smallest 1% of standardized distances.

In [ ]:
# x_obs: the observed 30 x 30 array, 1 = dry
def summaries(X):
    return np.array([X.mean(), (X[:, 1:] != X[:, :-1]).mean()])
s_obs = summaries(x_obs)
def sim_summaries(p, rho):
    return summaries(simulate_weather(p, rho))

# pilot: the scale of each summary under the prior, then frozen
pilot = np.array([sim_summaries(*rng.random(2))
                  for _ in range(2000)])
scales = pilot.std(axis=0)

B = 20_000
p = rng.random(B)
rho = rng.random(B)
S = np.array([sim_summaries(p[i], rho[i]) for i in range(B)])
d = np.sqrt((((S - s_obs) / scales)**2).sum(axis=1))
keep = d <= np.quantile(d, 0.01)   # closest 1%
post = np.column_stack([p[keep], rho[keep]])

## Sampling from the posterior predictive distribution

For $b=1,\ldots,R$: choose a retained pair $(p^{(b)},\rho^{(b)})$, then simulate a fresh dataset $X_{\mathrm{rep}}^{(b)}$ of 30 years from the rain model at that pair.

In [ ]:
R = 4000
reps = []
for r in range(R):
    i = rng.integers(len(post))   # one pair for the whole dataset
    reps.append(simulate_weather(post[i, 0], post[i, 1]))

## The slope statistic

$K_i$ is the number of weather switches in year $i$. Our statistic $T(X)$ is the least-squares slope of $K_i$ against year $i$. Does the fitted model commonly generate a trend as strong as the observed one?

In [ ]:
years = np.arange(1, 31)
def trend_slope(X):
    K = (X[:, 1:] != X[:, :-1]).sum(axis=1)   # switches per year
    return np.polyfit(years, K, 1)[0]
slopes = np.array([trend_slope(X) for X in reps])
plt.hist(slopes)
plt.axvline(trend_slope(x_obs))
np.mean(slopes <= trend_slope(x_obs))

## Passing posterior predictive checks does not establish identifiability

Return to the insect trap and its ABC posterior from 30 nights. Simulate fresh 30-night datasets, each at one retained $(\lambda,q)$ pair. Compare features we did not use for fitting:

- the variance of the nightly catches
- the number of nights with no catch

In [ ]:
# post_trap: retained (lam, q) pairs from the trap ABC fit
R = 4000
pick = rng.integers(len(post_trap), size=R)
# one row per replicated 30-night dataset
reps = np.array([simulate_catches(*post_trap[k], 30)
                 for k in pick])
rep_var = reps.var(axis=1, ddof=1)
rep_zero = (reps == 0).sum(axis=1)
fig, ax = plt.subplots(3, 4)   # observed and 11 replicates
ax[0, 0].hist(y_obs)
for r in range(11):
    ax.flat[r + 1].hist(reps[r])
fig, ax = plt.subplots(1, 2)
ax[0].hist(rep_var); ax[0].axvline(y_obs.var(ddof=1))
ax[1].hist(rep_zero); ax[1].axvline((y_obs == 0).sum())
print(np.mean(rep_var >= y_obs.var(ddof=1)))
print(np.mean(rep_zero >= (y_obs == 0).sum()))

Does the observed histogram stand out from the replicates?

# 3. How SMM and ABC use summaries

Suppose

$$
X_1,\ldots,X_n\overset{\mathrm{iid}}{\sim}N(\mu_0,\sigma^2).
$$

The population mean $\mu_0$ is known. We want to estimate $\sigma^2$. For this comparison, use only the sample mean $\bar X$ as the summary.

## Simulating sample means

In [ ]:
def simulate_means(sigma, n_sims, n=25, mu0=0):
    X = rng.normal(mu0, sigma, (n_sims, n))
    return X.mean(axis=1)    # one mean per simulated sample

sigmas = np.arange(0.5, 6.5, 0.5)
means = [simulate_means(s, 500) for s in sigmas]

fig, ax = plt.subplots()
ax.violinplot(means, positions=sigmas, widths=0.4, showmeans=True)
for s, m in zip(sigmas, means):
    jitter = rng.uniform(-0.1, 0.1, len(m))
    ax.scatter(s + jitter, m, s=1, color="gray")

The average of the simulated means stays at $\mu_0$ for every $\sigma$. Their spread grows with $\sigma$.

## ABC using the same sample mean

ABC retains a proposed variance if $|\bar X_{\mathrm{sim}}-\bar x_{\mathrm{obs}}|<\epsilon$. Draw variances from a prior, simulate samples, and retain those whose means match.

In [ ]:
B = 200_000
v = rng.uniform(0.01, 36, B)    # variance prior
# row i: a sample of 25 with standard deviation sqrt(v[i])
X = rng.normal(0, np.sqrt(v)[:, None], (B, 25))
xbar = X.mean(axis=1)
kept = v[np.abs(xbar - 0.4) < 0.05]
plt.hist(kept)

The retained distribution reflects how often each variance generates a matching mean, as well as the prior.